# Запуск сохранённой ResNet18
Этот ноутбук только загружает твои веса и распознаёт фотографию. Обучать модель заново не нужно.
Архитектура соответствует файлу `ResNet18_cat_or_dog_v1 (1).ipynb`: шаг первой свёртки 1, начальный MaxPool убран, последний слой — Linear с двумя выходами.

### 1. Библиотеки
Запусти ячейку. Если библиотеки ещё не установлены, выполни в отдельной ячейке `%pip install torch torchvision matplotlib pillow` и при необходимости перезапусти ядро.

In [ ]:
from pathlib import Path
import torch
import torchvision
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms
from PIL import Image

# Для одного фото достаточно CPU.
device = "cpu"


### 2. Файл весов
Положи `cat_dog_fixed.pt` в рабочую папку или укажи полный путь, например `~/Downloads/cat_dog_fixed.pt`. Нужны веса именно твоей адаптированной ResNet18, а не CNN или EfficientNet.

In [ ]:
weights_path = "cat_dog_fixed.pt"
weights_path = Path(weights_path).expanduser()

print("Рабочая папка:", Path.cwd())
if not weights_path.is_file():
    raise FileNotFoundError(f"Укажи правильный путь к весам: {weights_path}")


### 3. Загрузка модели
Создаём такую же архитектуру, как при обучении, и загружаем веса. Предобученные веса из интернета не скачиваются.

In [ ]:
model = torchvision.models.resnet18(weights=None)
model.conv1.stride = (1, 1)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

model.load_state_dict(
    torch.load(weights_path, map_location=device, weights_only=True)
)
model.eval()
print("Модель загружена. Можно проверять фотографии.")


### 4. Обработка фото и функция predict
Уменьшаем RGB-фото до 32×32 и нормализуем, как при обучении. Случайного отражения здесь нет: повторная проверка одного фото не меняет его обработку. Выход 0 — кошка, 1 — собака.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict(path):
    path = Path(path).expanduser()
    with Image.open(path) as image:
        photo = image.convert("RGB")

    tensor = transform(photo).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(tensor), dim=1)[0].cpu()

    return photo, {
        "cat": probabilities[0].item(),
        "dog": probabilities[1].item()
    }


### 5. Выбор фото и результат
Укажи путь к фотографии и запусти ячейку. Для другого фото достаточно поменять путь и повторить только эту ячейку.
Проценты — оценки модели. Она выбирает между кошкой и собакой даже для другого объекта. После перезапуска ядра выполни все ячейки сверху вниз.

In [ ]:
photo_path = "~/Downloads/photo.jpeg"

photo, result = predict(photo_path)

print(f"Кошка: {result['cat']:.2%}")
print(f"Собака: {result['dog']:.2%}")

if result["cat"] > result["dog"]:
    answer = f"Кошка — {result['cat']:.2%}"
elif result["dog"] > result["cat"]:
    answer = f"Собака — {result['dog']:.2%}"
else:
    answer = "Вероятности равны"

print("Итог:", answer)
plt.figure(figsize=(5, 5))
plt.imshow(photo)
plt.title(answer)
plt.axis("off")
plt.show()
